In [14]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

In [11]:
df = pd.read_csv("dataset/amazon_review.csv")
print(f"Raw Dataset: \n{df.head(1)}")

# Keep only the needed column
df = df[["reviewText", "overall"]]

# Remove the missing values (if exists)
print("")
print(df[["reviewText", "overall"]].isnull().sum())
df = df.dropna(subset=["reviewText","overall"])

print("")
print(f"Dataset: \n{df.head(1)}")

Raw Dataset: 
       reviewerID        asin reviewerName helpful  reviewText  overall  \
0  A3SBTW3WS4IQSN  B007WTAJTO          NaN  [0, 0]  No issues.      4.0   

      summary  unixReviewTime  reviewTime  day_diff  helpful_yes  total_vote  
0  Four Stars      1406073600  2014-07-23       138            0           0  

reviewText    1
overall       0
dtype: int64

Dataset: 
   reviewText  overall
0  No issues.      4.0


In [12]:
def convert_rating_to_label(rating): # Convert the rating to Negative, Neutral, and Positive
    if rating in [1, 2]:
        return 0        # Negative
    elif rating == 3:
        return 1        # Neutral
    else:
        return 2        # Positive

In [13]:
df["label"] = df["overall"].apply(convert_rating_to_label)
print(df.head(2))

                                          reviewText  overall  label
0                                         No issues.      4.0      2
1  Purchased this for my device, it worked as adv...      5.0      2


In [16]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df["reviewText"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("X_train size:", len(X_train))
print("X_test size:", len(X_test))
print("X_val size:", len(X_val))

X_train size: 3931
X_test size: 492
X_val size: 491


In [ ]:
train_set = Dataset.from_dict(
    {
        "reviewText": X_train.to_list(),
        "label": y_train.to_list()
    }
)

test_set = Dataset.from_dict(
    {
        "reviewText": X_test.to_list(),
        "label": y_test.to_list()
    }
)

val_set = Dataset.from_dict(
    {
        "reviewText": X_val.to_list(),
        "label": y_val.to_list()
    }
)

In [ ]:
model = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model)

def tokenizer_function(text):
    return tokenizer(
        text["reviewText"],
        padding=True,
        truncation=True,
        max_length=128
        )